In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score
from tqdm import tqdm
import torch.nn.functional as F
import random
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)


### 参数设置

In [2]:
IMG_SIZE = 448
BATCH_SIZE = 64
data_dir = "../data/aptos/"
base_output_dir = "./exp_effb4_regression"      # 输出文件夹
exp_msg = "con_loss"                            # 实验信息
os.environ['CUDA_VISIBLE_DEVICES'] = '0, 1, 5, 6'
if_bottleneck = False # 是否启用瓶颈层

lr=9e-5
scheduler_patience = 5
scheduler_facotr = 0.5
epochs = 22
patience = 6
reg_loss_weight = 1.0
cls_loss_weight = 1.0
consistency_loss_weight = 0.25

num_classes = 5
devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]

os.makedirs(base_output_dir, exist_ok=True)
# 自动获取下一个 exp{n} 序号
existing_exps = [d for d in os.listdir(base_output_dir) if d.startswith('exp') and d[3:].isdigit()]
next_id = max([int(d[3:]) for d in existing_exps], default=0) + 1
exp_dir = os.path.join(base_output_dir, f"exp{next_id}")
os.makedirs(exp_dir, exist_ok=True)
print(f"当前实验结果将保存至: {exp_dir}")

当前实验结果将保存至: ./exp_effb4_regression/exp4


### 读取数据

In [3]:
import numpy as np
import pandas as pd
from PIL import Image
from joblib import Parallel, delayed # 并行加速

image_dir = os.path.join(data_dir, "image")
X_cache_path = os.path.join(data_dir, f"X_preprocessed_{IMG_SIZE}.npy")
y_cache_path = os.path.join(data_dir, f"y_preprocessed_{IMG_SIZE}.npy")
csv_path = os.path.join(data_dir, "train.csv")
train_data = pd.read_csv(csv_path)

# 返回 PIL.Image
def load_one_image(image_id):
    img_path = os.path.join(image_dir, f"{image_id}.png")
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return img


def load_raw_data():
    if os.path.exists(X_cache_path) and os.path.exists(y_cache_path):
        print("检测到缓存文件，正在快速加载...")
        X = np.load(X_cache_path)
        y = np.load(y_cache_path)
    else:
        print("首次运行，正在处理图片（可能较慢）...")
        print("开启多进程并行读取...")
        X_list = Parallel(n_jobs=-1)(
            delayed(load_one_image)(image_id) for image_id in train_data['id_code']
        )
        X = np.array(X_list)
        y = train_data['diagnosis'].values
        # 保存原始图片
        np.save(X_cache_path, X)
        np.save(y_cache_path, y)
    return X, y


In [4]:
X, y = load_raw_data()

检测到缓存文件，正在快速加载...


In [5]:
import numpy as np
import pandas as pd
from PIL import Image
from joblib import Parallel, delayed # 并行加速
data_dir = "../data/aptos-2019/"
image_dir = os.path.join(data_dir, "val_images")
X_cache_path = os.path.join(data_dir, f"X_preprocessed_{IMG_SIZE}.npy")
y_cache_path = os.path.join(data_dir, f"y_preprocessed_{IMG_SIZE}.npy")
csv_path = os.path.join(data_dir, "valid.csv")
train_data = pd.read_csv(csv_path)

# 返回 PIL.Image
def load_one_image(image_id):
    img_path = os.path.join(image_dir, f"{image_id}.png")
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return img


def load_raw_data():
    if os.path.exists(X_cache_path) and os.path.exists(y_cache_path):
        print("检测到缓存文件，正在快速加载...")
        X = np.load(X_cache_path)
        y = np.load(y_cache_path)
    else:
        print("首次运行，正在处理图片（可能较慢）...")
        print("开启多进程并行读取...")
        X_list = Parallel(n_jobs=-1)(
            delayed(load_one_image)(image_id) for image_id in train_data['id_code']
        )
        X = np.array(X_list)
        y = train_data['diagnosis'].values
        # 保存原始图片
        np.save(X_cache_path, X)
        np.save(y_cache_path, y)
    return X, y

X_val, y_val = load_raw_data()

检测到缓存文件，正在快速加载...


### 定义数据集

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from imblearn.over_sampling import SMOTE

class DRDataset(Dataset):
    # images 应该是未归一化的图片
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        image, label = self.images[idx], self.labels[idx]
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        # 回归任务中标签需为浮点数
        return image, torch.tensor(label, dtype=torch.float) 


In [7]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05, hue=0.05),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),  
    transforms.RandomRotation(15),  
    transforms.ToTensor(), # 自动进行像素值归一化
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

X_train = X
y_train = y

from torch.utils.data import WeightedRandomSampler
# 重新计算针对全量数据的采样权重
class_counts = np.bincount(y_train)
print(f"全量数据集分布: {class_counts}")

class_weights = 1. / class_counts
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(y_train),
    replacement=True
)

train_loader = DataLoader(
    DRDataset(X_train, y_train, train_tf), 
    batch_size=BATCH_SIZE, 
    sampler=sampler, 
    num_workers=16
)
val_loader = DataLoader(
    DRDataset(X_val, y_val, val_tf), 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=16
)

全量数据集分布: [1434  300  808  154  234]


### 创建网络

In [8]:
def create_effb4_regressor():
    weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
    model = models.efficientnet_b4(weights=weights)
    in_features = model.classifier[1].in_features
    if if_bottleneck:
        model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.LayerNorm(256), # 关键：将特征约束在超球面上，有利于线性插值
        nn.ReLU(),
        nn.Linear(256, 1 + num_classes) # 1回归 + 5分类
        )
    else:
        model.classifier[1] = nn.Linear(in_features, 1 + num_classes)
    return model
model = create_effb4_regressor().to(devices[0])

### 多任务训练循环

In [9]:
criterion_reg = nn.SmoothL1Loss() # 回归
criterion_cls = nn.CrossEntropyLoss() # 分类
params_lx = [
    param 
    for name, param in model.named_parameters()
    if "classifier" not in name 
]
optimizer = optim.Adam(
    [{'params': params_lx}, {'params': model.classifier.parameters(), 'lr': lr * 10}], 
    lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=scheduler_facotr, patience=scheduler_patience)
if len(devices) > 1:
    model = nn.DataParallel(model, device_ids=range(len(devices)))

In [ ]:
best_kappa = -1.0
best_val_loss = float('inf') # 增加 loss 监控
counter = 0

print(f"开始多任务训练: 回归({reg_loss_weight}) + 分类({cls_loss_weight})")
for epoch in range(epochs):
    # 训练阶段
    model.train()
    train_loss, train_reg_loss, train_cls_loss = 0.0, 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(devices[0]), labels.to(devices[0])
        optimizer.zero_grad()
        
        outputs = model(imgs)
        # 拆分输出
        reg_out = outputs[:, 0]        # 第 0 列是回归
        cls_out = outputs[:, 1:]       # 第 1-5 列是分类

        loss_reg = criterion_reg(reg_out, labels.float())
        loss_cls = criterion_cls(cls_out, labels.long())

        loss = loss_reg * reg_loss_weight + loss_cls * cls_loss_weight
        loss.backward()
        optimizer.step()
        # 记录各部分损失
        train_loss += loss.item() * imgs.size(0)
        train_reg_loss += loss_reg.item() * imgs.size(0)
        train_cls_loss += loss_cls.item() * imgs.size(0)
        # 实时更新进度条信息
        pbar.set_postfix({
            'L': f"{loss.item():.3f}", 
            'reg': f"{loss_reg.item():.3f}", 
            'cls': f"{loss_cls.item():.3f}"
        })

    avg_train_loss = train_loss / len(X_train)
    avg_train_reg = train_reg_loss / len(X_train)
    avg_train_cls = train_cls_loss / len(X_train)

    # 验证阶段
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_reg_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(devices[0])
            labels = labels.to(devices[0])
            outputs = model(imgs)
            reg_out = outputs[:, 0]
            cls_out = outputs[:, 1:]
            loss = criterion_reg(reg_out, labels.float()) + criterion_cls(cls_out, labels.long())

            val_loss += loss.item() * imgs.size(0)
            val_reg_preds.append(reg_out.cpu().numpy())
            val_targets.append(labels.cpu().numpy())
            val_correct += (cls_out.argmax(1) == labels).sum().item()

    # 计算验证集指标
    avg_val_loss = val_loss / len(X_val)
    val_cls_acc = val_correct / len(X_val)
    
    val_reg_preds = np.concatenate(val_reg_preds)
    val_reg_targets = np.concatenate(val_targets)
    
    # 将回归预测值转换为整数标签（Round + Clip）
    final_preds = np.round(np.clip(val_reg_preds, 0, 4)).astype(int)
    final_targets = val_reg_targets.astype(int)
    
    # 计算回归分支的 Kappa 指标
    kappa = cohen_kappa_score(final_targets, final_preds, weights='quadratic')
    scheduler.step(avg_val_loss)
    # scheduler.step(kappa)
    # scheduler.step(val_acc)
    
    print(f"\n[Epoch {epoch+1}] ------------------------------------------")
    print(f"  Train | Loss: {avg_train_loss:.4f} (Reg: {avg_train_reg:.3f}, Cls: {avg_train_cls:.3f})")
    print(f"  Valid | Loss: {avg_val_loss:.4f} | Cls_Acc: {val_cls_acc:.4f} | Reg_Kappa: {kappa:.4f}")

    # 保存最佳模型
    if kappa > best_kappa:
        best_kappa = kappa
    # if avg_val_loss < best_val_loss:
    #     best_val_loss = avg_val_loss
        counter = 0
        save_path = os.path.join(exp_dir, "best_model_reg.pth")
        # 考虑 DataParallel 的保存方式 
        torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(), save_path)
        # 保存用于后期阈值搜索的 OOF 数据
        oof_df = pd.DataFrame({'diagnosis': final_targets, 'pred_reg': val_reg_preds})
        oof_df.to_csv(os.path.join(exp_dir, "val_oof_reg.csv"), index=False)
        print(f"  >>> Best Model Saved (Loss Improved)")
    else: 
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

开始多任务训练: 回归(1.0) + 分类(1.0)


Epoch 1/22 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]/home/gsintern/anaconda3/envs/vggt/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
Epoch 1/22 [Train]: 100%|██████████| 46/46 [00:22<00:00,  2.07it/s, L=1.634, reg=0.475, cls=1.160]



[Epoch 1] ------------------------------------------
  Train | Loss: 2.2753 (Reg: 0.823, Cls: 1.452)
  Valid | Loss: 1.3353 | Cls_Acc: 0.5765 | Reg_Kappa: 0.7638
  >>> Best Model Saved (Loss Improved)


Epoch 2/22 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]/home/gsintern/anaconda3/envs/vggt/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
Epoch 2/22 [Train]: 100%|██████████| 46/46 [00:22<00:00,  2.03it/s, L=1.590, reg=0.579, cls=1.012]



[Epoch 2] ------------------------------------------
  Train | Loss: 1.5029 (Reg: 0.397, Cls: 1.106)
  Valid | Loss: 0.9930 | Cls_Acc: 0.6639 | Reg_Kappa: 0.8231
  >>> Best Model Saved (Loss Improved)


...

Epoch 20/22 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]/home/gsintern/anaconda3/envs/vggt/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
Epoch 20/22 [Train]: 100%|██████████| 46/46 [00:21<00:00,  2.19it/s, L=0.556, reg=0.139, cls=0.417]



[Epoch 20] ------------------------------------------
  Train | Loss: 0.5324 (Reg: 0.142, Cls: 0.391)
  Valid | Loss: 0.6915 | Cls_Acc: 0.8224 | Reg_Kappa: 0.8837


Epoch 21/22 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]/home/gsintern/anaconda3/envs/vggt/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
Epoch 21/22 [Train]: 100%|██████████| 46/46 [00:22<00:00,  2.04it/s, L=0.404, reg=0.140, cls=0.264]



[Epoch 21] ------------------------------------------
  Train | Loss: 0.5041 (Reg: 0.144, Cls: 0.360)
  Valid | Loss: 0.6846 | Cls_Acc: 0.8197 | Reg_Kappa: 0.8831


Epoch 22/22 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]/home/gsintern/anaconda3/envs/vggt/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
Epoch 22/22 [Train]: 100%|██████████| 46/46 [00:20<00:00,  2.21it/s, L=0.679, reg=0.230, cls=0.449]


### 记录实验结果

In [ ]:
print(f"best_val_loss: {best_val_loss}\n")
# 计算最佳准确率并格式化文件名
save_filename = f"best_val_loss_{best_val_loss:.4f}.txt"
file_path = os.path.join(exp_dir, save_filename)

print(f"正在将历史指标写入: {file_path}")

with open(file_path, "w", encoding='utf-8') as f:
    pass
print("记录写入完成。")

best_val_loss: 0.6840828619693797

正在将历史指标写入: ./exp_effb4_regression/exp2/best_val_loss_0.6841.txt
记录写入完成。


In [ ]:
import shutil
current_notebook = "EfficientNet-B4_multi_task.ipynb"
if os.path.exists(current_notebook):
    shutil.copy(current_notebook, os.path.join(exp_dir, f"{current_notebook}_{exp_msg}.ipynb"))
    print(f"代码已备份至: {os.path.join(exp_dir, current_notebook)}_{exp_msg}.ipynb")
else:
    print(f"警告: 未找到文件 {current_notebook}，备份失败。请检查文件名是否正确。")

print(f"当前实验结果将保存至: {exp_dir}")

代码已备份至: ./exp_effb4_regression/exp2/EfficientNet-B4_multi_task.ipynb_con_loss.ipynb
当前实验结果将保存至: ./exp_effb4_regression/exp2
